In [1]:
import json
from pathlib import Path
import pandas as pd

# Project root (assuming notebook runs from notebooks/)
BASE_DIR = Path.cwd().parent

folder = BASE_DIR / "data" / "raw" / "train_positions"

# Find the latest JSON file
latest_file = max(
    folder.glob("*.json"),
    key=lambda p: p.stat().st_mtime
)

with open(latest_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print("File:", latest_file.name)
print("Number of train entities:", len(data["entity"]))

File: train_positions_20261027_090330_522854.json
Number of train entities: 238


In [2]:
df_trains = pd.json_normalize(data["entity"])

print("Shape:", df_trains.shape)
print("\nColumns:")
print(df_trains.columns.tolist())

display(df_trains.head())

Shape: (238, 12)

Columns:
['id', 'vehicle.trip.trip_id', 'vehicle.trip.schedule_relationship', 'vehicle.trip.route_id', 'vehicle.position.latitude', 'vehicle.position.longitude', 'vehicle.timestamp', 'vehicle.congestion_level', 'vehicle.stop_id', 'vehicle.vehicle.id', 'vehicle.vehicle.label', 'vehicle.occupancy_status']


,id,vehicle.trip.trip_id,vehicle.trip.schedule_relationship,vehicle.trip.route_id,vehicle.position.latitude,vehicle.position.longitude,vehicle.timestamp,vehicle.congestion_level,vehicle.stop_id,vehicle.vehicle.id,vehicle.vehicle.label,vehicle.occupancy_status
0,1,N410.1324.165.12.D.10.92000922,SCHEDULED,CCN_2a,-33.109630,151.48701,1793091567,UNKNOWN_CONGESTION_LEVEL,MetroNet.561,7086.6002.1846.9718.4636.2113.8979.9394.7450.5944,19:21 Newcastle Interchange Station to Central...,NaN
1,2,8976.1324.165.8.W.0.92004721,SCHEDULED,RTTA_DEF,-33.709538,150.31633,1793090570,UNKNOWN_CONGESTION_LEVEL,Mountains.KT10/21 Loc,3874,NaN,NaN
2,3,W667.1324.165.44.D.10.92001717,SCHEDULED,BMT_1,-33.582930,150.25812,1793089072,UNKNOWN_CONGESTION_LEVEL,Mountains.MV50 Loc,1729.7376.7581.6303.5153.4080.6744.2589.1077.2402,16:26 Central Station to Katoomba Station,NaN
3,4,R761.1324.165.12.D.6.92003182,SCHEDULED,RTTA_REV,-33.863810,151.06140,1793088654,UNKNOWN_CONGESTION_LEVEL,Lidcombe.ST329 Loc,1623.6714.4505.2115.7559.8411,18:54 Central Station to Central Station,NaN
4,5,W474.1324.165.12.D.10.92002212,SCHEDULED,BMT_2,-33.721560,150.60304,1793091687,UNKNOWN_CONGESTION_LEVEL,Mountains.Warrimoo 1 Loc,7179.4653.8449.9642.4807.5809.6649.2763.7792.4225,18:52 Mount Victoria Station to Central Station,NaN


In [3]:
df_train_clean = df_trains[[
    "vehicle.trip.trip_id",
    "vehicle.trip.route_id",
    "vehicle.position.latitude",
    "vehicle.position.longitude",
    "vehicle.timestamp",
    "vehicle.stop_id",
    "vehicle.vehicle.label"
]].copy()

df_train_clean = df_train_clean.rename(columns={
    "vehicle.trip.trip_id": "trip_id",
    "vehicle.trip.route_id": "route_id",
    "vehicle.position.latitude": "latitude",
    "vehicle.position.longitude": "longitude",
    "vehicle.timestamp": "timestamp",
    "vehicle.stop_id": "stop_id",
    "vehicle.vehicle.label": "train_label"
})

# Convert Unix timestamps to readable UTC dates
df_train_clean["timestamp"] = pd.to_datetime(
    df_train_clean["timestamp"],
    unit="s",
    utc=True,
    errors="coerce"
)

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,N410.1324.165.12.D.10.92000922,CCN_2a,-33.109630,151.48701,NaT,MetroNet.561,19:21 Newcastle Interchange Station to Central...
1,8976.1324.165.8.W.0.92004721,RTTA_DEF,-33.709538,150.31633,NaT,Mountains.KT10/21 Loc,NaN
2,W667.1324.165.44.D.10.92001717,BMT_1,-33.582930,150.25812,NaT,Mountains.MV50 Loc,16:26 Central Station to Katoomba Station
3,R761.1324.165.12.D.6.92003182,RTTA_REV,-33.863810,151.06140,NaT,Lidcombe.ST329 Loc,18:54 Central Station to Central Station
4,W474.1324.165.12.D.10.92002212,BMT_2,-33.721560,150.60304,NaT,Mountains.Warrimoo 1 Loc,18:52 Mount Victoria Station to Central Station


In [4]:
df_train_clean["timestamp"] = pd.to_datetime(
    pd.to_numeric(df_train_clean["timestamp"], errors="coerce"),
    unit="s",
    utc=True,
    errors="coerce"
)

# Convert UTC to Sydney time
df_train_clean["timestamp"] = (
    df_train_clean["timestamp"]
    .dt.tz_convert("Australia/Sydney")
)

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,N410.1324.165.12.D.10.92000922,CCN_2a,-33.109630,151.48701,NaT,MetroNet.561,19:21 Newcastle Interchange Station to Central...
1,8976.1324.165.8.W.0.92004721,RTTA_DEF,-33.709538,150.31633,NaT,Mountains.KT10/21 Loc,NaN
2,W667.1324.165.44.D.10.92001717,BMT_1,-33.582930,150.25812,NaT,Mountains.MV50 Loc,16:26 Central Station to Katoomba Station
3,R761.1324.165.12.D.6.92003182,RTTA_REV,-33.863810,151.06140,NaT,Lidcombe.ST329 Loc,18:54 Central Station to Central Station
4,W474.1324.165.12.D.10.92002212,BMT_2,-33.721560,150.60304,NaT,Mountains.Warrimoo 1 Loc,18:52 Mount Victoria Station to Central Station


In [5]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
folder = BASE_DIR / "data" / "raw" / "train_positions"

latest_file = max(folder.glob("*.json"), key=lambda p: p.stat().st_mtime)

with open(latest_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Recreate DataFrame from original data
df_trains = pd.json_normalize(data["entity"])

# Check the original timestamp values
print(df_trains["vehicle.timestamp"].head())

0    1793091567
1    1793090570
2    1793089072
3    1793088654
4    1793091687
Name: vehicle.timestamp, dtype: str


In [6]:
# Start fresh from the original DataFrame
df_train_clean["timestamp"] = df_trains["vehicle.timestamp"]

# Convert strings to numbers
df_train_clean["timestamp"] = pd.to_numeric(
    df_train_clean["timestamp"],
    errors="coerce"
)

# Convert Unix seconds to Sydney local time
df_train_clean["timestamp"] = pd.to_datetime(
    df_train_clean["timestamp"],
    unit="s",
    utc=True
).dt.tz_convert("Australia/Sydney")

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,N410.1324.165.12.D.10.92000922,CCN_2a,-33.109630,151.48701,2026-10-27 19:59:27+11:00,MetroNet.561,19:21 Newcastle Interchange Station to Central...
1,8976.1324.165.8.W.0.92004721,RTTA_DEF,-33.709538,150.31633,2026-10-27 19:42:50+11:00,Mountains.KT10/21 Loc,NaN
2,W667.1324.165.44.D.10.92001717,BMT_1,-33.582930,150.25812,2026-10-27 19:17:52+11:00,Mountains.MV50 Loc,16:26 Central Station to Katoomba Station
3,R761.1324.165.12.D.6.92003182,RTTA_REV,-33.863810,151.06140,2026-10-27 19:10:54+11:00,Lidcombe.ST329 Loc,18:54 Central Station to Central Station
4,W474.1324.165.12.D.10.92002212,BMT_2,-33.721560,150.60304,2026-10-27 20:01:27+11:00,Mountains.Warrimoo 1 Loc,18:52 Mount Victoria Station to Central Station


In [7]:
# Calculate how old each train position is
now = pd.Timestamp.now(tz="Australia/Sydney")

df_train_clean["minutes_old"] = (
    (now - df_train_clean["timestamp"])
    .dt.total_seconds() / 60
)

# Show the freshest positions first
display(
    df_train_clean[
        ["train_label", "route_id", "timestamp", "minutes_old"]
    ].sort_values("minutes_old").head(10)
)

,train_label,route_id,timestamp,minutes_old
150,19:42 Leppington Station to Schofields Station,CMB_2a,2026-10-27 20:03:15+11:00,7.853841
209,NaN,NaN,2026-10-27 20:03:13+11:00,7.887175
164,19:47 Central Station to Sydenham Station,APS_1e,2026-10-27 20:03:13+11:00,7.887175
174,19:33 Cronulla Station to Bondi Junction Station,ESI_2d,2026-10-27 20:03:12+11:00,7.903841
184,19:14 Central Station to Revesby Station,APS_1a,2026-10-27 20:03:12+11:00,7.903841
199,19:41 Central Station to Leppington Station,IWL_1c,2026-10-27 20:03:12+11:00,7.903841
191,19:50 Central Station to Macarthur Station,APS_1a,2026-10-27 20:03:11+11:00,7.920508
120,18:41 Central Station to Kiama Station,SCO_1a,2026-10-27 20:03:11+11:00,7.920508
103,19:15 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,7.920508
173,19:45 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,7.920508


In [8]:
# Get the collection time from the filename
snapshot_time = pd.to_datetime(
    latest_file.stem.split("_")[-3] + "_" +
    latest_file.stem.split("_")[-2] + "_" +
    latest_file.stem.split("_")[-1],
    format="%Y%m%d_%H%M%S_%f",
    utc=True
).tz_convert("Australia/Sydney")

# Compare each train's timestamp with collection time
df_train_clean["minutes_old_at_collection"] = (
    (snapshot_time - df_train_clean["timestamp"])
    .dt.total_seconds() / 60
)

print("Snapshot collected:", snapshot_time)

display(
    df_train_clean[
        ["train_label", "route_id", "timestamp",
         "minutes_old_at_collection"]
    ].sort_values("minutes_old_at_collection").head(10)
)

Snapshot collected: 2026-10-27 20:03:30.522854+11:00


,train_label,route_id,timestamp,minutes_old_at_collection
150,19:42 Leppington Station to Schofields Station,CMB_2a,2026-10-27 20:03:15+11:00,0.258714
209,NaN,NaN,2026-10-27 20:03:13+11:00,0.292048
164,19:47 Central Station to Sydenham Station,APS_1e,2026-10-27 20:03:13+11:00,0.292048
174,19:33 Cronulla Station to Bondi Junction Station,ESI_2d,2026-10-27 20:03:12+11:00,0.308714
184,19:14 Central Station to Revesby Station,APS_1a,2026-10-27 20:03:12+11:00,0.308714
199,19:41 Central Station to Leppington Station,IWL_1c,2026-10-27 20:03:12+11:00,0.308714
191,19:50 Central Station to Macarthur Station,APS_1a,2026-10-27 20:03:11+11:00,0.325381
120,18:41 Central Station to Kiama Station,SCO_1a,2026-10-27 20:03:11+11:00,0.325381
103,19:15 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,0.325381
173,19:45 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,0.325381


In [9]:
# Convert position age into seconds
df_train_clean["seconds_old"] = (
    df_train_clean["minutes_old_at_collection"] * 60
)

print(df_train_clean["seconds_old"].describe())

print("\nPositions within 30 seconds:",
      df_train_clean["seconds_old"].between(0, 30).sum())

print("Positions within 60 seconds:",
      df_train_clean["seconds_old"].between(0, 60).sum())

print("Positions older than 5 minutes:",
      (df_train_clean["seconds_old"] > 300).sum())

count       238.000000
mean       8796.174115
std       62718.200926
min          15.522854
25%          39.522854
50%          81.022854
75%         496.022854
max      873322.522854
Name: seconds_old, dtype: float64

Positions within 30 seconds: 38
Positions within 60 seconds: 102
Positions older than 5 minutes: 73


In [12]:
import folium
import pandas as pd

train_map = folium.Map(
    location=[-33.8688, 151.2093],
    zoom_start=10,
    tiles="OpenStreetMap"
)

for _, train in df_train_clean.iterrows():

    if pd.isna(train["latitude"]) or pd.isna(train["longitude"]):
        continue

    age = train["seconds_old"]

    if pd.isna(age) or age < 0:
        continue
    elif age <= 60:
        colour = "green"
    elif age <= 300:
        colour = "orange"
    else:
        colour = "gray"

    label = train["train_label"]

    if pd.isna(label):
        label = "Unknown train"

    popup = f"""
    <b>Train:</b> {label}<br>
    <b>Route:</b> {train['route_id']}<br>
    <b>Last position:</b> {train['timestamp']}<br>
    <b>Age at collection:</b> {age:.0f} seconds
    """

    folium.CircleMarker(
        location=[train["latitude"], train["longitude"]],
        radius=5,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.8,
        popup=folium.Popup(popup, max_width=350)
    ).add_to(train_map)

train_map.save("sydney_train_map.html")

In [13]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

gtfs_dir = BASE_DIR / "data" / "raw" / "gtfs_static" / "gtfs_static"

routes = pd.read_csv(gtfs_dir / "routes.txt", dtype=str)
trips = pd.read_csv(gtfs_dir / "trips.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)

display(routes.head())

Routes: (10237, 9)
Trips: (211419, 11)


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,exact_times
0,1-SC0-1-sj2-1,701,SC01,Kiama to Bomaderry,Illawarra Buses Network,700,00B5EF,FFFFFF,1
1,1-SH1-0-sj2-2,701,SH100,Moss Vale to Goulburn,Illawarra Buses Network,700,00B5EF,FFFFFF,1
2,11-400-1-sj2-1,2501,4001,Penrith Station to McCarthy College,School buses,712,00B5EF,FFFFFF,1
3,11-400-2-sj2-1,2501,4002,Swallow Dr opp Skylark Cres to Caroline Chisho...,School buses,712,00B5EF,FFFFFF,1
4,11-400-3-sj2-1,2501,4003,Caroline Chisholm College to York PS via Regen...,School buses,712,00B5EF,FFFFFF,1


In [14]:
train_routes = routes[routes["agency_id"] == "x0001"].copy()

print("Number of Sydney Trains routes:", len(train_routes))

display(
    train_routes[
        ["route_short_name", "route_long_name", "route_type"]
    ]
)

Number of Sydney Trains routes: 10


,route_short_name,route_long_name,route_type
2320,T1,T1 North Shore and Western Line,2
2321,T1,T1 North Shore and Western Line,2
2322,T2,T2 Leppington and Inner West Line,2
2323,T3,T3 Liverpool and Inner West Line,2
2324,T4,T4 Eastern Suburbs and Illawarra Line,2
2325,T5,T5 Cumberland Line,2
2326,T6,T6 Lidcombe and Bankstown Line,2
2327,T7,T7 Olympic Park Line,2
2328,T8,T8 Airport and South Line,2
2329,T9,T9 Northern Line,2


In [15]:
train_trips = trips.merge(
    train_routes[["route_id", "route_short_name"]],
    on="route_id",
    how="inner"
)

shape_ids = set(train_trips["shape_id"].dropna())

print("Total Sydney Trains trips:", len(train_trips))
print("Unique railway shapes:", len(shape_ids))

display(
    train_trips[
        ["route_short_name", "trip_id", "shape_id"]
    ].head(10)
)

Total Sydney Trains trips: 48298
Unique railway shapes: 1298


,route_short_name,trip_id,shape_id
0,T1,101A.2010.103.128.T.8.91040144,2-T1-N-sj2-1.13.R
1,T1,101A.2010.103.2.T.8.91040144,2-T1-N-sj2-1.13.R
2,T1,101D.2010.103.128.T.8.91040148,2-T1-N-sj2-1.23.H
3,T1,101D.2010.103.2.T.8.91040148,2-T1-N-sj2-1.23.H
4,T1,101E.2010.103.128.T.8.91040150,2-T1-N-sj2-1.9.R
5,T1,101E.2010.103.2.T.8.91040150,2-T1-N-sj2-1.9.R
6,T1,101H.2010.103.128.T.8.91040156,2-T1-N-sj2-1.23.H
7,T1,101H.2010.103.2.T.8.91040156,2-T1-N-sj2-1.23.H
8,T1,101J.2010.103.128.T.8.91040158,2-T1-N-sj2-1.9.R
9,T1,101J.2010.103.2.T.8.91040158,2-T1-N-sj2-1.9.R


In [16]:
train_shape_chunks = []

for chunk in pd.read_csv(
    gtfs_dir / "shapes.txt",
    dtype={"shape_id": str},
    chunksize=100_000
):
    selected = chunk[
        chunk["shape_id"].isin(shape_ids)
    ]

    if not selected.empty:
        train_shape_chunks.append(selected)

train_shapes = pd.concat(
    train_shape_chunks,
    ignore_index=True
)

train_shapes["shape_pt_sequence"] = pd.to_numeric(
    train_shapes["shape_pt_sequence"]
)

train_shapes = train_shapes.sort_values(
    ["shape_id", "shape_pt_sequence"]
)

print("Total railway coordinates:", len(train_shapes))
print("Shapes extracted:", train_shapes["shape_id"].nunique())

display(train_shapes.head())

Total railway coordinates: 852072
Shapes extracted: 1298


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,2-T1-N-sj2-1.1.R,-33.623511,151.153121,1,0.00
1,2-T1-N-sj2-1.1.R,-33.624566,151.151528,2,188.53
2,2-T1-N-sj2-1.1.R,-33.624789,151.151271,3,222.85
3,2-T1-N-sj2-1.1.R,-33.625196,151.150681,4,293.81
4,2-T1-N-sj2-1.1.R,-33.625464,151.150281,5,341.41


In [18]:
output_dir = BASE_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_shapes.to_parquet(
    output_dir / "sydney_train_shapes.parquet",
    index=False
)

print("Railway shapes saved!")

Railway shapes saved!
